# Step 2: Eigenpair Error Analysis

Bound a candidate eigenpair's residual, correct its eigenvalue to make the residual orthogonal to its velocity, and bound the normalized velocity and gradient. These provide the paper's inputs `eq:45-input-Ev`, `eq:45-input-lambda`, `eq:45-input-v`, and `eq:45-input-grad-v`.

Run from `notebooks/` with `dtype = Interval{Float64}` selected before loading the module.

**Inputs:** `data/generated/UP_divfree.mat` from Step 1 and the raw candidate and `lambda` in `data/external/up_eig.mat`.

**Output:** `data/generated/up_eig_corrected.mat`, used by Steps 5, 6, and 7. It encloses the exactly normalized velocity/pressure candidate, corrected eigenvalue, scale, and evaluation-grid shape. The file is written only when absent; an existing export must correspond to the same input fields and normalization calculation.

| Notebook check | Paper input |
|---|---|
| Normalized residual upper endpoint ≤ `4.808e-6` | $\varepsilon^v\le4.81\times10^{-6}$ |
| Corrected eigenvalue upper endpoint ≤ `-0.1131623` | $\overline\lambda\le-0.1131$ |
| Normalized velocity L∞ upper endpoint ≤ `0.0318488` | $\|\overline v\|_\infty\le0.0319$ |
| Normalized gradient L∞ upper endpoint ≤ `0.0431148` | $\|\nabla\overline v\|_\infty\le0.0432$ |

Read in order: load fields → global derivative priors → physical-domain refinement → norm and eigenvalue correction → uniform bounds → normalization and export. The [symmetry reference](../docs/technical_notes/symmetry_and_domains.md#eigenpair-verification) explains the domains.


In [1]:
import Pkg
Pkg.activate("..")
include("../src/NS_Numerics.jl")
using .NS_Numerics

  Activating 

Using dtype = IntervalArithmetic

project at `~/Changhe/3d-navier-stokes-nonuniqueness-main-run`


.Interval{Float64}
Number of threads = 56


## Load the background and candidate eigenpair

`UP_divfree.mat` contains the prepared background profile. Its loader uses boundary set 2 and `divfree=true`, which skips a second divergence-free reconstruction. The candidate is loaded from the original `up_eig.mat` using the field-odd boundary set 1. The candidate loader's default reconstructs its angular velocity component to enforce the divergence-free relation.

Let $\kappa=\texttt{UP.scl\_fac}$, $r_{\rm phy}=\kappa\tan\beta$, and let $\widehat v,\widehat q$ be the stored candidate velocity and pressure. The physical fields are $v=\cos\beta\,\widehat v$ and $q=\cos\beta\,\widehat q$. The operator scale factors supply physical derivatives. With $D=(I+x\cdot\nabla)/2$, define the physical residual at the loaded eigenvalue $\lambda=\texttt{lbd}$ by

$$
E_\lambda=(U\cdot\nabla)v+(v\cdot\nabla)U-Dv-\Delta v-\nabla q-\lambda v.
$$

The code uses the stationary elliptic sign convention; the corresponding evolution generator has the opposite eigenvalue sign. The pressure above has the stored sign. See [the operator and pressure conventions](../docs/technical_notes/ns_terms.md#9-get_linear-weighted-physical-linear-terms).

The radial weight is $w(\beta)=\sin\beta/\cos^2\beta$. The evaluated objects have distinct meanings:

| Object | Represented field | Use |
|---|---|---|
| `err` | $wE_\lambda$ | Residual derivatives and physical L² norm |
| `I_up` | $wv$ | Physical L² normalization and correction integrals |
| `I_phy_up` | $v$ | Physical vector L∞ bound |
| `grad_up` | Nine entries of the physical $\nabla v$ | Gradient L∞ bound |

The weighted arrays are integration variables. They must not be confused with the physical fields used in the L∞ estimates.

In [2]:
# Load the prepared background; divfree=true skips repeating its divergence-free reconstruction.
UP = from_mat_file("../data/generated/UP_divfree.mat", 2; domain="frequency", divfree=true)
# UP = from_mat_file("../data/external/UP.mat", 2; domain="frequency", divfree=false)
M0_UP, M1_UP = size(UP["u3"].freq_func)

# Load the original candidate and stored eigenvalue; the default loader reconstructs divergence-free u2.
up, lbd = from_mat_file("../data/external/up_eig.mat", 1; domain="frequency")
M0, M1 = size(up["u3"].freq_func)
println("eigenvalue: ", lbd)

# Target residual refinement order; the coarse pass supplies two higher orders.
k = 8;


eigenvalue: [-0.113163, -0.113162]_com


## Full-period priors and the field-odd symmetries

The first β grid is the **analytic trigonometric extension** $[0,2\pi]$, not a larger physical radial region. It supplies global suprema for Bernstein estimates. Physical space is still represented by $\beta\in[0,\pi/2)$.

The candidate's stored `u1,u2` retain a special $l=1$ radial `10` block and higher-degree `00` blocks. Its `sine_even` pressure uses radial `00` for every angular column, with angular `10` and no Legendre conversion. Their radial reflection signs differ, and the assembled residual can mix both signs at the endpoints. A radial quarter-domain maximum need not be a full-period maximum; the full-period prior avoids this assumption.

Angular reduction remains valid. The field-odd vector components have signs $(+,-,-)$ under $\theta\mapsto-\theta$ and $(-,+,-)$ under $\theta\mapsto\pi-\theta$ (codes `10,01,00`). The linearized residual and H preserve this sector, and every full-gradient entry also has definite angular reflection signs. Absolute values of their angular derivatives therefore attain their full-period suprema on $[0,\pi/2]$.

| Stage | β interval | θ interval | Interior intervals | Padding per side |
|---|---|---|---|---|
| Coarse frequency priors | $[0,2\pi]$ | $[0,\pi/2]$ | `32000 × 4000` | 0 |
| Residual refinement and subsequent integrals | $[0,\pi/2]$ | $[0,\pi/2]$ | `36000 × 18000` | 10 for refinement; removed before integration |

`N0,N1` count intervals; an unpadded grid has `(N0+1) × (N1+1)` samples. The refinement grid has `36021 × 18021` samples before trimming.

`norm_dict` stores derivative bounds through order ten for `err` and `I_up`, and componentwise L∞ bounds for `I_phy_up` and `grad_up`. For a derivative table `W`, `W[m+1,j][i]` bounds derivative order m of component i, with direction `j=1` for β and `j=2` for θ. Coarse tables have shape `11 × 2` with three-component entries; the refined residual table has shape `9 × 2`.

The candidate-only frequency ceilings are `[2*M0+10, 2*M1+10]`, covering `I_up`, physical velocity, and physical gradient. Mixed UP/candidate convection in `err` instead uses `[2*(M0_UP+M0)+10, 2*(M1_UP+M1)+10]`; the same residual ceiling is used in both the coarse priors and refinement. These ceilings are computed from the coefficient dimensions of the actual loaded fields. These are conservative upper ceilings, not exact nonzero frequencies. The frequency branch first corrects sampled maxima to continuous bounds and then applies Bernstein powers. See the frequency accounting in the symmetry reference. The raw field is normalized by its exact physical L² norm, enclosed by `u_L2`.

In [3]:
norm_dict, err_freq_lst = let
    # Full periodic β extension for Bernstein priors; the physical radial domain remains [0,π/2].

    # Full β period and symmetry-reduced angular domain.
    L0, L1 = Pi * dt(2), Pi / dt(2)
    # Interior interval counts (one more unpadded sample per axis).
    N0, N1 = 32000, 4000

    # Unpadded uniform grid for the trigonometric prior evaluation.
    β_pt = grid(L0, N0)
    θ_pt = grid(L1, N1)

    # lin_mat returns w*(Dv+Δv+∇q); I_mat returns w*v.
    lin_mat = get_operator_matrix(up, β_pt, θ_pt, "linear")
    I_mat = get_operator_matrix(up, β_pt, θ_pt, "identity")
    grad_mat = get_operator_matrix(up, β_pt, θ_pt, "gradient")
    I_phy_mat = get_operator_matrix(up, β_pt, θ_pt, "identity"; weight_flag=false)

    # Weighted transport terms.
    U_grad_u = mul_A_gradB(UP, up, β_pt, θ_pt)
    u_grad_U = mul_A_gradB(up, UP, β_pt, θ_pt)

    # Assemble weighted fields for physical L² estimates: err=w*E_λ.
    I_up = apply(I_mat, up) 
    err = U_grad_u + u_grad_U - apply(lin_mat, up) - I_up * lbd

    # Unweighted physical velocity and full nine-entry gradient for L∞ bounds.
    grad_up = apply(grad_mat, up; new_keys=["A11", "A12", "A13", "A21", "A22", "A23", "A31", "A32", "A33"])
    I_phy_up = apply(I_phy_mat, up; new_keys=["u1", "u2", "u3"]);

    # Candidate-only ceilings; mixed residual terms use both spectral resolutions.
    freq_lst = dt.([2 * M0 + 10, 2 * M1 + 10])
    err_freq_lst = dt.([2 * (M0_UP + M0) + 10, 2 * (M1_UP + M1) + 10])

    # Keep the evaluated fields associated with their derivative-bound dictionary keys.
    vec_dict = Dict("err" => err, "I_up" => I_up, "grad_up" => grad_up, "I_phy_up" => I_phy_up)
    norm_dict = Dict()

    # Orders 0:10 seed refinement and quadrature; physical velocity/gradient need componentwise L∞ bounds.
    for key in keys(vec_dict)
        mode = key in ["I_up", "err"] ? "W$(k+2)Inf" : "LInf"
        freqs = key == "err" ? err_freq_lst : freq_lst
        norm_dict[key] = norm_lst(vec_dict[key], mode; paras=freqs, domain=[L0,L1])
    end
    # These bounds refer to the unnormalized input; later reports divide by the common u_L2.

    (norm_dict, err_freq_lst)
end
GC.gc(true)


## Refine the residual on the physical domain

The next block rebuilds `err` and `I_up` on the physical β/θ rectangle, keeping ten ghost layers per side. The original `lbd` is used throughout this residual evaluation. `derivative_padding(8)=10` supplies the nested stencils for bounds through order eight, with coarse order-nine and order-ten bounds as seeds. The `domain` lengths remain those of the unpadded grid, so padding does not alter the mesh spacing.

The call `Linf_to_paras(...; ind=1)` refines β first from explicit priors, then constructs angular bounds from the refined order-zero and order-two β bounds. If these are $B_0,B_2$, angular Bernstein gives

$$
\|\partial_\theta^m f\|_\infty\le M_\theta^mB_0,\qquad
\|\partial_\beta^2\partial_\theta^m f\|_\infty\le M_\theta^mB_2.
$$

These provide the θ seeds and transverse correction without applying β Bernstein on the restricted physical interval. The angular reflection signs remain valid after β differentiation and extend the bounds to the angular ghost samples. 

The samples retain cancellation between the assembled residual terms; the priors control the finite-difference error. Trim both arrays only after the derivative bounds have been obtained. See [the special refinement construction](../docs/technical_notes/matrix_func.md#special-mode-refine-β-first-then-construct-the-θ-bounds).

## Physical integrals and the eigenvalue correction

After refinement, both arrays are trimmed to the unpadded physical grid. The inner-product routine includes $\sin\theta$ and $4\pi\kappa^3$:

$$
\langle f,g\rangle_{L^2(\mathbb R^3)}
=4\pi\kappa^3\int_0^{\pi/2}\!\int_0^{\pi/2}
(wf)\cdot(wg)\sin\theta\,d\theta\,d\beta.
$$

Azimuthal integration supplies $2\pi$, and same-sector equatorial symmetry supplies another factor 2. The radial interval already covers all nonnegative physical radii; no radial doubling is used. Supplying `W_norms_lst` adds the eighth-derivative Newton–Cotes error allowance, including the same physical measure factor.

Assume $\|v\|_{L^2}>0$. The code computes enclosures of

$$
\texttt{err\_v}=\langle E_\lambda,v\rangle,\qquad
\texttt{v\_v}=\|v\|_{L^2}^2,\qquad
\lambda_* = \lambda + \frac{\langle E_\lambda,v\rangle}{\|v\|_{L^2}^2}.
$$

Thus `λ_new = lbd + err_v/v_v` has a **plus** sign: the residual contains $-\lambda v$. With $c=\langle E_\lambda,v\rangle/\|v\|_{L^2}^2$,

$$
E_{\lambda_*}=E_\lambda-cv,\qquad
\langle E_{\lambda_*},v\rangle=0,\qquad
\|E_{\lambda_*}\|_2^2
=\|E_\lambda\|_2^2-\frac{\langle E_\lambda,v\rangle^2}{\|v\|_{L^2}^2}.
$$

`λ_new` encloses the exact corrected value $\lambda_*$. The orthogonality identity concerns that value; interval arithmetic need not return a literal zero interval after substitution.

**How to interpret the residual output.** The code still integrates `err` at the original `lbd`; it does not rebuild the corrected residual. Nevertheless,

$$
\|E_{\lambda_*}\|_2\le\|E_\lambda\|_2.
$$

Therefore the **upper endpoint** of `err_L2/u_L2` also bounds the corrected residual of the exactly normalized field in `eq:45-input-Ev`. The displayed interval's lower endpoint is not a lower bound for the corrected residual. `λ_new` supplies the eigenvalue enclosure for `eq:45-input-lambda`. The eigenvalue correction requires `v_v` to have a strictly positive lower endpoint. The interval `u_L2 = sqrt(v_v)` encloses the exact physical norm; the reported bounds and exported velocity/pressure are divided by this common factor.

These are physical function-space inner products, not coefficient dot products. See the [quadrature derivation](../docs/technical_notes/matrix_func.md#from-one-dimensional-quadrature-to-a-weighted-inner-product).

In [4]:
err_L2, u_L2, λ_new = let
    # Rebuild weighted residual samples on the physical domain, with actual stencil padding.
    L0, L1 = Pi / dt(2), Pi / dt(2)
    N0, N1 = 36000, 18000

    # Extend grids for recursive order-eight refinement, retaining ten ghost layers.
    padding = derivative_padding(k)
    β_pt = grid(L0, N0, padding)
    θ_pt = grid(L1, N1, padding)

    # Re-evaluate both weighted convection terms; keep the loaded eigenvalue lbd.
    err = mul_A_gradB(up, UP, β_pt, θ_pt) + mul_A_gradB(UP, up, β_pt, θ_pt)

    lin_mat = get_operator_matrix(up, β_pt, θ_pt, "linear")
    I_mat = get_operator_matrix(up, β_pt, θ_pt, "identity")

    # I_up=w*v; subtract the weighted base operator and original eigenvalue term.
    I_up = apply(I_mat, up) 
    minus_equal!(err, apply(lin_mat, up) + I_up * lbd);

    # ind=1 uses explicit β seeds, then angular Bernstein with refined β-order-0/2 bounds.
    paras_lst = Linf_to_paras(norm_dict["err"], k+2, err_freq_lst; ind=1)
    norm_dict["err"] = norm_lst(err, "W$(k)Inf"; paras=paras_lst, domain=[L0,L1], padding=padding)

    # Remove ghost layers only after refinement; quadrature uses the physical unpadded grid.
    trim_border!(err, padding)
    trim_border!(I_up, padding)

    # Physical inner products, with derivative-based quadrature allowances.
    err_v = inner_product(err, I_up; W_norms_lst=[norm_dict["err"], norm_dict["I_up"]], domain=[L0, L1])
    v_v = inner_product(I_up, I_up; W_norms_lst=[norm_dict["I_up"], norm_dict["I_up"]], domain=[L0, L1])
    @assert inf(v_v) > 0 "Physical squared norm must be positive"
    # Plus sign removes the residual component along v because E_λ contains -λ*v.
    λ_new = lbd + err_v / v_v
    println("λ new: ", λ_new)
    @assert sup(λ_new) <= inf(dt("-0.1131623"))

    # This is the original residual norm; its upper endpoint also bounds the corrected residual norm.
    err_L2 = norm_lst(err, "L2"; paras=norm_dict["err"])
    u_L2 = sqrt(v_v)
    epsilon_v = err_L2 / u_L2
    println("L^2 norm of error: ", epsilon_v)
    @assert sup(epsilon_v) <= inf(dt("4.808e-6"))

    (err_L2, u_L2, λ_new)
end
GC.gc(true)


λ new: [-0.113163, -0.113162]_com
L^2 norm of error: [4.27525e-6, 4.80778e-6]_com


## Adjoint source for Step 7

The auxiliary-field calculation uses

$$
H(v)=Dv-\Delta v-(\overline U\cdot\nabla)v+(\nabla\overline U)^Tv,
\qquad D=(I+x\cdot\nabla)/2.
$$

The `"transpose"` operator branch returns $w(\Delta-D)v$ without pressure. The paper's source in `eq:eigenvalue-functional-ibp` is $h_0=H(v)-v/2$. Since the test fields are orthogonal to the normalized v,

$$
P_vh_0=P_vH(v),\qquad P_vF=F-\langle F,v\rangle v.
$$

[Step 7](verify_auxiliary_field_bound.ipynb) estimates the composed functional $(I+(G^v)^*)h_0$ in the weighted energy dual norm. This notebook supplies the normalized v and residual bound used there. See [the adjoint derivation](../docs/technical_notes/ns_terms.md#10-get_linear-modetranspose-and-the-projected-quantity).


In [5]:
# Standalone adjoint L² norm omitted; verify_auxiliary_field_bound.ipynb bounds ℓ₀ ∘ B.

## Physical L∞ bounds after normalization

The full-period pass already bounded the physical velocity components and all nine physical gradient entries. Each bound is divided by the same $\|v\|_{L^2}$ used above, assigned, printed as an interval, and immediately checked against its target.

If $b_i$ bounds the ith velocity component and $c_{ij}$ bounds each gradient entry, then

$$
\left\|\frac{v}{\|v\|_{L^2}}\right\|_{L^\infty}\le\frac{\sqrt{\sum_i b_i^2}}{\|v\|_{L^2}},\qquad
\left\|\nabla\left(\frac{v}{\|v\|_{L^2}}\right)\right\|_{L^\infty(F)}\le\frac{\sqrt{\sum_{i,j}c_{ij}^2}}{\|v\|_{L^2}}.
$$

These are the expressions printed from `I_phy_up` and `grad_up`. The gradient bound uses the Frobenius matrix norm and also bounds the matrix operator norm. All nine entries are included, so no packed-six-component factor is needed. Combining component suprema this way is conservative: their maxima need not occur at the same point. The upper endpoints must satisfy `v_Linf ≤ inf(dt("0.0318488"))` and `grad_v_Linf ≤ inf(dt("0.0431148"))`; the code checks these immediately after their respective print calls.

In [6]:
# Combine normalized component bounds: vector Euclidean norm and full-gradient Frobenius norm.
v_Linf = norm(norm_dict["I_phy_up"] / u_L2)
println("Linf v: ", v_Linf)
@assert sup(v_Linf) <= inf(dt("0.0318488"))

grad_v_Linf = norm(norm_dict["grad_up"] / u_L2)
println("Linf grad v: ", grad_v_Linf)
@assert sup(grad_v_Linf) <= inf(dt("0.0431148"))

Linf v: [0.0318486, 0.0318487]_com
Linf grad v: [0.0431146, 0.0431147]_com


## Normalize and save

Divide velocity and pressure by the common interval `u_L2` enclosing the exact physical velocity norm, then save the coefficients and `λ_new`. The intervals enclose the same exactly normalized candidate and corrected eigenvalue used in the estimates above. The unit-norm and orthogonality identities refer to that candidate, rather than every independent choice of interval endpoints.

The export is written only when absent. Steps 5–7 must use an export from the same raw eigenpair, background, and normalization/correction calculation.


In [7]:
divide_equal!(up, u_L2)
if !isfile("../data/generated/up_eig_corrected.mat")
    save(up, "../data/generated/up_eig_corrected.mat"; lambda=λ_new)
end
;